In [9]:
!pip install numpy scipy imutils opencv-python

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/39.0 MB ? eta -:--:--
   - -------------------------------------- 1.0/39.0 MB 7.1 MB/s eta 0:00:06
   - -------------------------------------- 1.6/39.0 MB 4.6 MB/s eta 0:00:09
   -- ------------------------------------- 2.1/39.0 MB 3.8 MB/s eta 0:00:10
   -- ------------------------------------- 2.9/39.0 MB 3.9 MB/s eta 0:00:10
   --- ------------------------------------ 3.7/39.0 MB 3.6 MB/s eta 0:00:10
   ---- ----------------------------------- 4.2/39.0 MB 3.5 MB/s eta 0:00:10
   ---- ----------------------------------- 4.7/39.0 MB 3.4 MB/s eta 0:00:10
   ----- ---------------------------------- 5.2/39.0 MB 3.3 MB/s eta 0:00:11
   ------ --------------------------------- 6.0/39.0 MB 3.2 MB/s eta 0:00:11
   ------ --------------------------------- 6.6/39.0 MB 3.2 MB/s eta 0:00:11
   ------- -------------------------------- 7.1/39.0 MB 3.

  DEPRECATION: Building 'imutils' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'imutils'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  You can safely remove it manually.
  You can safely remove it manually.


In [7]:
!pip install scipy

In [7]:
import social_distancing_config as config
from social_distancing_config import MIN_CONF
from scipy.spatial import distance as dist
import numpy as np
import argparse
import imutils
import cv2
import os


ap = argparse.ArgumentParser()
ap.add_argument("-i", "--input", type=str, default="",
	help="path to (optional) input video file")
ap.add_argument("-o", "--output", type=str, default="",
	help="path to (optional) output video file")
ap.add_argument("-d", "--display", type=int, default=1,
	help="whether or not output frame should be displayed")
args = vars(ap.parse_args(["--input","D:/social-distance-detector/crowded.mp4","--output","my_output.avi","--display","1"]))



labelsPath = "yolo-coco/coco.names"
LABELS = open(labelsPath).read().strip().split("\n")


weightsPath = "yolo-coco/yolov3.weights"
configPath = "yolo-coco/yolov3.cfg"


print("[INFO] loading YOLO from disk...")
net = cv2.dnn.readNetFromDarknet(configPath, weightsPath)

if config.USE_GPU:
	# set CUDA as the preferable backend and target
	print("[INFO] setting preferable backend and target to CUDA...")
	net.setPreferableBackend(cv2.dnn.DNN_BACKEND_CUDA)
	net.setPreferableTarget(cv2.dnn.DNN_TARGET_CUDA)


ln = net.getLayerNames()
try:
    # for OpenCV 3.x/older 4.x
    ln = [ln[i[0] - 1] for i in net.getUnconnectedOutLayers()]
except:
    # for OpenCV 4.3+
    ln = [ln[i - 1] for i in net.getUnconnectedOutLayers()]


print("[INFO] accessing video stream...")
vs = cv2.VideoCapture(args["input"] if args["input"] else 0)
writer = None

# loop over the frames from the video stream
while True:
	# read the next frame from the file
	(grabbed, frame) = vs.read()


	if not grabbed:
		break

	frame = imutils.resize(frame, width=700)
	results = detect_people(frame, net, ln,
		personIdx=LABELS.index("person"))

	
	violate = set()


	if len(results) >= 2:
		
		centroids = np.array([r[2] for r in results])
		D = dist.cdist(centroids, centroids, metric="euclidean")

		# loop over the upper triangular of the distance matrix
		for i in range(0, D.shape[0]):
			for j in range(i + 1, D.shape[1]):
				
				if D[i, j] < config.MIN_DISTANCE:
					# update our violation set with the indexes of
					# the centroid pairs
					violate.add(i)
					violate.add(j)


	# loop over the results
	for (i, (prob, bbox, centroid)) in enumerate(results):
		# extract the bounding box and centroid coordinates, then
		# initialize the color of the annotation
		(startX, startY, endX, endY) = bbox
		(cX, cY) = centroid
		color = (0, 255, 0)

		
		
		if i in violate:
			color = (0, 0, 255)

		
		
		cv2.rectangle(frame, (startX, startY), (endX, endY), color, 2)
		cv2.circle(frame, (cX, cY), 5, color, 1)


	text = "Social Distancing Violations: {}".format(len(violate))
	cv2.putText(frame, text, (10, frame.shape[0] - 25),
		cv2.FONT_HERSHEY_SIMPLEX, 0.85, (0, 0, 255), 3)



	
	if args["display"] > 0:
		# show the output frame
		cv2.imshow("Frame", frame)
		key = cv2.waitKey(1) & 0xFF

		if key == ord("q"):
			break


	if args["output"] != "" and writer is None:
		# initialize our video writer
		fourcc = cv2.VideoWriter_fourcc(*"MJPG")
		writer = cv2.VideoWriter(args["output"], fourcc, 25,
			(frame.shape[1], frame.shape[0]), True)

	
	if writer is not None:
		writer.write(frame)


[INFO] loading YOLO from disk...
[INFO] accessing video stream...
